# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed0607/ML-Internship-Strter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The Grain (One Row =): A single pseudonymized content item's aggregated performance metrics for the month of March 2026.

Tables Used: The warehouse fact_content_daily_performance table (filtered to month=2026-03), joined with dim_content to bring in metadata.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Time Window: A single mid-panel month (month=2026-03) to ensure a stable snapshot without hitting the volatile newest 3 days.

Target/Proxy: Because this is an unsupervised clustering task, the target proxy is the Cluster ID assigned by our algorithm, representing the performance archetype.

Deliberately Excluded: We are deliberately excluding product-derived context flags like health_score or priority_score to avoid a circular result, and raw article text, ensuring we cluster strictly on observed behavior and structural metadata.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
import duckdb
import pandas as pd
import os
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

df = pd.read_csv('content_refresh_anonymized.csv')
con.register('content_data', df)

print("FACT 1: THE GRAIN")
grain_check = con.execute("""
    SELECT content_id, COUNT(*) as row_count
    FROM content_data
    GROUP BY content_id
    ORDER BY row_count DESC
    LIMIT 3
""").df()
print(grain_check)
print("\n")
print("FACT 2: ROW COUNT & DATE SPAN")
span_check = con.execute("""
    SELECT COUNT(*) as total_rows, MIN(content_age_days) as min_age, MAX(content_age_days) as max_age
    FROM content_data
""").df()
print(span_check)
print("\n")
print("FACT 3: AVAILABILITY")
avail_check = con.execute("""
    SELECT COUNT(*) as active_pages
    FROM content_data
    WHERE impressions_90d > 0 IS TRUE
""").df()
print(avail_check)
print("\n")
print("THE 5-FEATURE FRAME")
feature_frame = df[['content_id', 'impressions_90d', 'sessions_90d', 'content_age_days', 'word_count']].dropna().copy()
print(feature_frame.head(3))
print("\n")
print("THE LEAKAGE TRAP")
feature_frame['dummy_target_is_high_traffic'] = (feature_frame['impressions_90d'] > 1000).astype(int)

X_leaked = feature_frame[['impressions_90d', 'sessions_90d', 'content_age_days']]
y = feature_frame['dummy_target_is_high_traffic']

model = DecisionTreeClassifier(random_state=42)
model.fit(X_leaked, y)
predictions = model.predict(X_leaked)
score = accuracy_score(y, predictions)
print(f"Accuracy WITH leaked target data: {score * 100}% (Too perfect, it's a trap!)")

X_honest = feature_frame[['sessions_90d', 'content_age_days', 'word_count']]
model.fit(X_honest, y)
honest_predictions = model.predict(X_honest)
honest_score = accuracy_score(y, honest_predictions)
print(f"Accuracy AFTER deleting the leak: {honest_score * 100:.2f}% (The honest number)")


FACT 1: THE GRAIN
             content_id  row_count
0  content_a1fb4e703a9e          1
1  content_9aa793d4d895          1
2  content_d99b7a2d90ca          1


FACT 2: ROW COUNT & DATE SPAN
   total_rows  min_age  max_age
0       30000       90      564


FACT 3: AVAILABILITY
   active_pages
0         30000


THE 5-FEATURE FRAME
             content_id  impressions_90d  sessions_90d  content_age_days  \
0  content_304f48230142             3803            17               187   
1  content_a1fb4e703a9e            15320             9               445   
2  content_9aa793d4d895            12581            11               141   

   word_count  
0      3221.0  
1      2481.0  
2      3515.0  


THE LEAKAGE TRAP
Accuracy WITH leaked target data: 100.0% (Too perfect, it's a trap!)
Accuracy AFTER deleting the leak: 99.90% (The honest number)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One named limitation of our slice:

  We are limited to observing behavioral metrics and structural metadata (like word count and age). Because the data contract deliberately excludes raw text to protect privacy, we have a "semantic blind spot." We can cluster pages that behave the exact same way, but we cannot mathematically confirm if they cover the same exact topic.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.